<a href="https://colab.research.google.com/github/SRIJANRAOS/srijanraos_INFO5731_spring2026/blob/main/Srijanraos_INFO5731_Assignment_1_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INFO5731 Assignment 1**

In this assignment, you will work on gathering text data from an open data source via web scraping or API. Following this, you will need to clean the text data and perform syntactic analysis on the data. Follow the instructions carefully and design well-structured Python programs to address each question.

**Expectations**:
*   Use the provided .*ipynb* document to write your code & respond to the questions. Avoid generating a new file.
*   Write complete answers and run all the cells before submission.
*   Make sure the submission is "clean"; *i.e.*, no unnecessary code cells.
*   Once finished, allow shared rights from top right corner (*see Canvas for details*).

* **Make sure to submit the cleaned data CSV in the comment section - 10 points**

**Total points**: 100


**Late Submission will have a penalty of 10% reduction for each day after the deadline.**

**Please check that the link you submitted can be opened and points to the correct assignment.**


# Question 1 (25 points)

Write a python program to collect text data from **either of the following sources** and save the data into a **csv file:**

(1) Collect all the customer reviews of a product (you can choose any porduct) on amazon. [atleast 1000 reviews]

(2) Collect the top 1000 User Reviews of a movie recently in 2024 or 2025 (you can choose any movie) from IMDB. [If one movie doesn't have sufficient reviews, collect reviews of atleast 2 or 3 movies]


(3) Collect the **abstracts** of the top 10000 research papers by using the query "machine learning", "data science", "artifical intelligence", or "information extraction" from Semantic Scholar.

(4) Collect all the information of the 904 narrators in the Densho Digital Repository.

(5)**Collect a total of 10000 reviews** of the top 100 most popular software from G2 and Capterra.


In [ ]:
# Your code here
import requests
import pandas as pd
import time

In [ ]:
import time
import requests
import pandas as pd

BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
QUERY = "machine learning"
TOTAL_PAPERS = 10000
LIMIT = 1000  # bulk supports up to 1000 per page :contentReference[oaicite:1]{index=1}

# OPTIONAL: If you have an API key, paste it here (helps with rate limits)
API_KEY = ""  # leave empty if you don't have one

headers = {
    "Accept": "application/json",
    "User-Agent": "CS-assignment (contact: your-email@example.com)"
}
if API_KEY:
    headers["x-api-key"] = API_KEY  # API key goes in header :contentReference[oaicite:2]{index=2}

rows = []
token = None

def get_with_retries(url, params, headers, tries=6):
    """Retry on server errors (like 500) with exponential backoff."""
    for attempt in range(tries):
        r = requests.get(url, params=params, headers=headers, timeout=30)
        if r.status_code < 500:
            r.raise_for_status()
            return r
        # 5xx (server error): wait and retry
        wait = 2 ** attempt
        print(f"Server error {r.status_code}. Retry in {wait}s...")
        time.sleep(wait)
    r.raise_for_status()  # if still failing after retries

while len(rows) < TOTAL_PAPERS:
    params = {
        "query": QUERY,
        "fields": "paperId,title,year,abstract,url",
        "limit": LIMIT
    }
    if token:
        params["token"] = token  # bulk pagination uses token :contentReference[oaicite:3]{index=3}

    response = get_with_retries(BASE_URL, params, headers)
    data = response.json()

    for paper in data.get("data", []):
        rows.append({
            "paperId": paper.get("paperId"),
            "title": paper.get("title"),
            "year": paper.get("year"),
            "abstract": paper.get("abstract"),
            "url": paper.get("url")
        })

    print("Collected:", len(rows))

    token = data.get("token")
    if not token:
        break

    time.sleep(1.0)  # small polite delay (increase if you get 429)

Collected: 1000
Collected: 2000
Collected: 3000
Collected: 4000
Collected: 5000
Collected: 6000
Collected: 7000
Collected: 8000
Collected: 8795


In [ ]:
df = pd.DataFrame(rows).head(TOTAL_PAPERS)
df.to_csv("semantic_scholar_raw_abstracts.csv", index=False, encoding="utf-8")
df.head()

,paperId,title,year,abstract,url
0,00000c33779acab142af6c7a6dae8b36fac0805d,Insights into Household Electric Vehicle Charg...,2024.0,In the era of burgeoning electric vehicle (EV)...,https://www.semanticscholar.org/paper/00000c33...
1,0000238f07f151172cf2602588ba762b55c8464b,Personalized Prediction of Response to Smartph...,2021.0,Background Meditation apps have surged in popu...,https://www.semanticscholar.org/paper/0000238f...
2,00002d31a8c758062a51d9a259313d81a5eaf399,A Machine Learning Method to Quantify the Role...,2020.0,None,https://www.semanticscholar.org/paper/00002d31...
3,0000315635be19f6278dbc72597b3065fac405f0,Abstractive text summarization of low-resource...,2023.0,Background Humans must be able to cope with th...,https://www.semanticscholar.org/paper/00003156...
4,00005d68c6c7eb4d3c27da8242a30b9a498f991e,Detection of DDoS Attacks on Clouds Computing ...,2023.0,The growing number of cloud-based services has...,https://www.semanticscholar.org/paper/00005d68...


# Question 2 (15 points)

Write a python program to **clean the text data** you collected in the previous question and save the clean data in a new column in the csv file. The data cleaning steps include: [Code and output is required for each part]

(1) Remove noise, such as special characters and punctuations.

(2) Remove numbers.

(3) Remove stopwords by using the stopwords list.

(4) Lowercase all texts

(5) Stemming.

(6) Lemmatization.

In [ ]:
# Write code for each of the sub parts with proper comments.

import pandas as pd

# Load the raw CSV you created in Q1
df = pd.read_csv("semantic_scholar_raw_abstracts.csv")

# Keep only rows where abstract is not missing
df = df[df["abstract"].notna()].copy()

# Show a few raw abstracts (OUTPUT)
df[["paperId", "abstract"]].head(3)

,paperId,abstract
0,00000c33779acab142af6c7a6dae8b36fac0805d,In the era of burgeoning electric vehicle (EV)...
1,0000238f07f151172cf2602588ba762b55c8464b,Background Meditation apps have surged in popu...
3,0000315635be19f6278dbc72597b3065fac405f0,Background Humans must be able to cope with th...


In [ ]:
import re
import nltk

# Download required NLTK data
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print("Stopwords loaded:", len(stop_words))  # OUTPUT

Stopwords loaded: 198


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# (1) Remove noise: keep letters, numbers, and spaces only (punctuation removed)
df["step1_no_punct"] = df["abstract"].apply(
    lambda x: re.sub(r"[^A-Za-z0-9\s]", " ", x)  # remove special chars/punct
)

# OUTPUT: before vs after
df[["abstract", "step1_no_punct"]].head(3)

,abstract,step1_no_punct
0,In the era of burgeoning electric vehicle (EV)...,In the era of burgeoning electric vehicle EV ...
1,Background Meditation apps have surged in popu...,Background Meditation apps have surged in popu...
3,Background Humans must be able to cope with th...,Background Humans must be able to cope with th...


In [ ]:
# (2) Remove numbers
df["step2_no_numbers"] = df["step1_no_punct"].apply(
    lambda x: re.sub(r"\d+", " ", x)  # remove digits
)

# OUTPUT: show result
df[["step1_no_punct", "step2_no_numbers"]].head(3)

,step1_no_punct,step2_no_numbers
0,In the era of burgeoning electric vehicle EV ...,In the era of burgeoning electric vehicle EV ...
1,Background Meditation apps have surged in popu...,Background Meditation apps have surged in popu...
3,Background Humans must be able to cope with th...,Background Humans must be able to cope with th...


In [ ]:
# (4) Lowercase all text
df["step4_lower"] = df["step2_no_numbers"].str.lower()

# OUTPUT
df[["step2_no_numbers", "step4_lower"]].head(3)

,step2_no_numbers,step4_lower
0,In the era of burgeoning electric vehicle EV ...,in the era of burgeoning electric vehicle ev ...
1,Background Meditation apps have surged in popu...,background meditation apps have surged in popu...
3,Background Humans must be able to cope with th...,background humans must be able to cope with th...


In [ ]:
# (3) Remove stopwords using NLTK stopword list
def remove_stopwords(text):
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df["step3_no_stopwords"] = df["step4_lower"].apply(remove_stopwords)

# OUTPUT
df[["step4_lower", "step3_no_stopwords"]].head(3)

,step4_lower,step3_no_stopwords
0,in the era of burgeoning electric vehicle ev ...,era burgeoning electric vehicle ev popularity ...
1,background meditation apps have surged in popu...,background meditation apps surged popularity r...
3,background humans must be able to cope with th...,background humans must able cope huge amounts ...


In [ ]:
# (5) Stemming (PorterStemmer)
def stem_text(text):
    return " ".join(stemmer.stem(w) for w in text.split())

df["step5_stem"] = df["step3_no_stopwords"].apply(stem_text)

# OUTPUT
df[["step3_no_stopwords", "step5_stem"]].head(3)

,step3_no_stopwords,step5_stem
0,era burgeoning electric vehicle ev popularity ...,era burgeon electr vehicl ev popular understan...
1,background meditation apps surged popularity r...,background medit app surg popular recent year ...
3,background humans must able cope huge amounts ...,background human must abl cope huge amount inf...


In [ ]:
# (6) Lemmatization (WordNetLemmatizer)
# Note: Without POS tags, WordNetLemmatizer assumes nouns by default.
def lemmatize_text(text):
    return " ".join(lemmatizer.lemmatize(w) for w in text.split())

df["step6_lemma"] = df["step3_no_stopwords"].apply(lemmatize_text)

# OUTPUT
df[["step3_no_stopwords", "step6_lemma"]].head(3)

,step3_no_stopwords,step6_lemma
0,era burgeoning electric vehicle ev popularity ...,era burgeoning electric vehicle ev popularity ...
1,background meditation apps surged popularity r...,background meditation apps surged popularity r...
3,background humans must able cope huge amounts ...,background human must able cope huge amount in...


In [ ]:
# Final cleaned text column
df["cleaned_text"] = df["step6_lemma"].str.replace(r"\s+", " ", regex=True).str.strip()

# Save to a new CSV for submission
out_file = "semantic_scholar_cleaned_abstracts.csv"
df.to_csv(out_file, index=False, encoding="utf-8")

print("Saved:", out_file)  # OUTPUT
df[["abstract", "cleaned_text"]].head(3)  # OUTPUT

Saved: semantic_scholar_cleaned_abstracts.csv


,abstract,cleaned_text
0,In the era of burgeoning electric vehicle (EV)...,era burgeoning electric vehicle ev popularity ...
1,Background Meditation apps have surged in popu...,background meditation apps surged popularity r...
3,Background Humans must be able to cope with th...,background human must able cope huge amount in...


# Question 3 (15 points)

Write a python program to **conduct syntax and structure analysis of the clean text** you just saved above. The syntax and structure analysis includes:

(1) **Parts of Speech (POS) Tagging:** Tag Parts of Speech of each word in the text, and calculate the total number of N(oun), V(erb), Adj(ective), Adv(erb), respectively.

(2) **Constituency Parsing and Dependency Parsing:** print out the constituency parsing trees and dependency parsing trees of all the sentences. Using one sentence as an example to explain your understanding about the constituency parsing tree and dependency parsing tree.

(3) **Named Entity Recognition:** Extract all the entities such as person names, organizations, locations, product names, and date from the clean texts, calculate the count of each entity.

In [ ]:
# Your code here
!pip -q install stanza
import stanza

# Download English models (run once)
stanza.download("en")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.7/773.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 19.0 MB/s eta 0:00:00


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources


[['zip', 'default.zip']]

In [ ]:
import pandas as pd

df = pd.read_csv("semantic_scholar_cleaned_abstracts.csv")

# Use the cleaned text column you created in Q2
texts = df["cleaned_text"].dropna().tolist()

print("Documents:", len(texts))
print("Example cleaned text:\n", texts[0][:300])

Documents: 5717
Example cleaned text:
 era burgeoning electric vehicle ev popularity understanding pattern ev user behavior imperative paper examines trend household charging session timing duration energy consumption analyzing real world residential charging data leveraging information collected session novel framework introduced effici


In [ ]:
nlp = stanza.Pipeline(
    lang="en",
    processors="tokenize,pos,lemma,constituency,depparse,ner",
    tokenize_no_ssplit=False
)

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor    | Package                   |
--------------------------------------------
| tokenize     | combined                  |
| mwt          | combined                  |
| pos          | combined_charlm           |
| lemma        | combined_nocharlm         |
| constituency | ptb3-revised_charlm       |
| depparse     | combined_charlm           |
| ner          | ontonotes-ww-multi_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: constituency
INFO:stanza:Loading: depparse
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


In [ ]:
import stanza
from collections import Counter

# POS-only pipeline (much faster)
nlp_pos = stanza.Pipeline("en", processors="tokenize,pos", use_gpu=False, verbose=False)

In [ ]:
from collections import Counter
import random

# Count POS on a sample to keep runtime reasonable
# Increase SAMPLE_DOCS if you want (e.g., 2000). 1000 is usually fine.
SAMPLE_DOCS = 1000

# Random sample of your cleaned texts
sample_texts = random.sample(texts, k=min(SAMPLE_DOCS, len(texts)))

counts = Counter({"Noun": 0, "Verb": 0, "Adj": 0, "Adv": 0})

for i, t in enumerate(sample_texts, start=1):
    doc = nlp_pos(t)
    for sent in doc.sentences:
        for w in sent.words:
            upos = w.upos
            if upos in ("NOUN", "PROPN"):
                counts["Noun"] += 1
            elif upos in ("VERB", "AUX"):
                counts["Verb"] += 1
            elif upos == "ADJ":
                counts["Adj"] += 1
            elif upos == "ADV":
                counts["Adv"] += 1

    if i % 50 == 0:
        print("Processed:", i)

counts

Processed: 50
Processed: 100
Processed: 150
Processed: 200
Processed: 250
Processed: 300
Processed: 350
Processed: 400
Processed: 450
Processed: 500
Processed: 550
Processed: 600
Processed: 650
Processed: 700
Processed: 750
Processed: 800
Processed: 850
Processed: 900
Processed: 950
Processed: 1000


Counter({'Noun': 81934, 'Verb': 20905, 'Adj': 23126, 'Adv': 4244})

In [ ]:
PRINT_N = 3   # change to a bigger number; or set to None to print ALL sentences

printed = 0

for t in texts:
    doc = nlp(t)
    for sent in doc.sentences:
        # Stop if we printed enough
        if PRINT_N is not None and printed >= PRINT_N:
            break

        print("\nSENTENCE:", sent.text)

        # Constituency tree
        print("\nConstituency tree:")
        print(sent.constituency)

        # Dependency parse (simple: word -> head with relation)
        print("\nDependency edges (word -> head, relation):")
        for w in sent.words:
            head = "ROOT" if w.head == 0 else sent.words[w.head - 1].text
            print(f"{w.text:15} -> {head:15} ({w.deprel})")

        printed += 1

    if PRINT_N is not None and printed >= PRINT_N:
        break


SENTENCE: era burgeoning electric vehicle ev popularity understanding pattern ev user behavior imperative paper examines trend household charging session timing duration energy consumption analyzing real world residential charging data leveraging information collected session novel framework introduced efficient real time prediction important charging characteristic utilizing historical data user specific feature machine learning model trained predict connection duration charging duration charging demand time next session model enhance understanding ev user behavior provide practical tool optimizing ev charging infrastructure effectively managing charging demand transportation sector becomes increasingly electrified work aim empower stakeholder insight reliable model enabling anticipate localized demand contribute sustainable integration electric vehicle grid

Constituency tree:
(ROOT (S (NP (NN era) (VBG burgeoning) (NML (NML (NML (NML (JJ electric) (NN vehicle)) (NN ev) (NN populari

In [ ]:
from collections import Counter

entity_type_counts = Counter()
entity_text_counts = Counter()  # optional: counts of actual entity strings

for t in texts:
    doc = nlp(t)
    for ent in doc.ents:
        entity_type_counts[ent.type] += 1
        entity_text_counts[ent.text] += 1

print("Entity type counts:")
entity_type_counts

# **Following Questions must answer using AI assitance**

#Question 4 (20 points).

Q4. (PART-1)
Web scraping data from the GitHub Marketplace to gather details about popular actions. Using Python, the process begins by sending HTTP requests to multiple pages of the marketplace (1000 products), handling pagination through dynamic page numbers. The key details extracted include the product name, a short description, and the URL.

 The extracted data is stored in a structured CSV format with columns for product name, description, URL, and page number. A time delay is introduced between requests to avoid server overload. ChatGPT can assist by helping with the parsing of HTML, error handling, and generating reports based on the data collected.

 The goal is to complete the scraping within a specified time limit, ensuring that the process is efficient and adheres to GitHub’s usage guidelines.

(PART -2)

1.   **Preprocess Data**: Clean the text by tokenizing, removing stopwords, and converting to lowercase.

2. Perform **Data Quality** operations.


Preprocessing:
Preprocessing involves cleaning the text by removing noise such as special characters, HTML tags, and unnecessary whitespace. It also includes tasks like tokenization, stopword removal, and lemmatization to standardize the text for analysis.

Data Quality:
Data quality checks ensure completeness, consistency, and accuracy by verifying that all required columns are filled and formatted correctly. Additionally, it involves identifying and removing duplicates, handling missing values, and ensuring the data reflects the true content accurately.


Github MarketPlace page:
https://github.com/marketplace?type=actions

In [ ]:
import time, random, re
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

In [ ]:
!pip -q install beautifulsoup4 lxml tqdm

In [ ]:
BASE = "https://github.com"
LIST_URL = "https://github.com/marketplace?type=actions"

session = requests.Session()
session.headers.update({
    # A realistic User-Agent helps prevent instant blocking
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})

In [ ]:
def parse_marketplace_actions(html: str, page_num: int):
    soup = BeautifulSoup(html, "lxml")

    rows = []

    # Robust strategy:
    # - Find links that look like /marketplace/actions/<slug>
    # - For each link, grab link text as name
    # - Grab a nearby description (often the next <p> after the heading container)
    for a in soup.select('a[href^="/marketplace/actions/"]'):
        name = a.get_text(strip=True)
        href = a.get("href", "")
        if not name or not href:
            continue

        url = urljoin(BASE, href)

        # Try to find a description near the link:
        # go up to a reasonable parent container then search for the first <p>
        desc = ""
        container = a.find_parent(["h3", "div", "article", "li"])
        if container:
            p = container.find_next("p")
            if p:
                desc = p.get_text(" ", strip=True)

        # Filter out obvious non-listing links (rare but possible)
        # Marketplace listing names are usually not empty and descriptions are usually non-trivial
        rows.append({
            "product_name": name,
            "description": desc,
            "url": url,
            "page": page_num
        })

    # De-duplicate within a page by URL (some pages can contain repeated links in nav/other areas)
    dedup = {}
    for r in rows:
        dedup[r["url"]] = r
    return list(dedup.values())

In [ ]:
def fetch_page(page_num: int, max_retries: int = 3, timeout: int = 30):
    params = {"type": "actions", "page": page_num}
    url = "https://github.com/marketplace"

    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(url, params=params, timeout=timeout)

            # If rate limited or blocked, back off and retry
            if r.status_code in (429, 403):
                sleep_s = 5 + attempt * 5 + random.random() * 2
                print(f"[page {page_num}] status {r.status_code} -> backing off {sleep_s:.1f}s")
                time.sleep(sleep_s)
                continue

            r.raise_for_status()
            return r.text

        except requests.RequestException as e:
            sleep_s = 2 + attempt * 2 + random.random()
            print(f"[page {page_num}] error: {e} -> retry in {sleep_s:.1f}s")
            time.sleep(sleep_s)

    return None

In [ ]:
TARGET_PRODUCTS = 1000
EST_PER_PAGE = 20
MAX_PAGES = int((TARGET_PRODUCTS / EST_PER_PAGE) + 5)  # ~55 as buffer

all_rows = []
seen_urls = set()

for page in tqdm(range(1, MAX_PAGES + 1)):
    html = fetch_page(page)
    if html is None:
        print(f"Stopping early: failed to fetch page {page}")
        break

    rows = parse_marketplace_actions(html, page)

    # add only new urls
    new = 0
    for r in rows:
        if r["url"] not in seen_urls:
            all_rows.append(r)
            seen_urls.add(r["url"])
            new += 1

    # polite delay (avoid server overload)
    time.sleep(1.0 + random.random() * 1.5)

    # stop if we have enough
    if len(all_rows) >= TARGET_PRODUCTS:
        break

print("Collected:", len(all_rows))

  0%|          | 0/55 [00:00<?, ?it/s]

Collected: 1015


In [ ]:
df = pd.DataFrame(all_rows, columns=["product_name", "description", "url", "page"])
df.to_csv("github_marketplace_actions.csv", index=False)
df.head(10)

,product_name,description,url,page
0,TruffleHog OSS,Find and verify leaked credentials in your sou...,https://github.com/marketplace/actions/truffle...,1
1,Metrics embed,An infographics generator with 40+ plugins and...,https://github.com/marketplace/actions/metrics...,1
2,yq - portable yaml processor,"create, read, update, delete, merge, validate ...",https://github.com/marketplace/actions/yq-port...,1
3,Super-Linter,Super-linter is a ready-to-run collection of l...,https://github.com/marketplace/actions/super-l...,1
4,Rebuild Armbian and Kernel,"Support Amlogic, Rockchip and Allwinner boxes",https://github.com/marketplace/actions/rebuild...,1
5,Gosec Security Checker,Runs the gosec security checker,https://github.com/marketplace/actions/gosec-s...,1
6,Checkout,Checkout a Git repository at a particular version,https://github.com/marketplace/actions/checkout,1
7,OpenCommit — improve commits with AI 🧙,Replaces lame commit messages with meaningful ...,https://github.com/marketplace/actions/opencom...,1
8,SSH Remote Commands,Executing remote ssh commands,https://github.com/marketplace/actions/ssh-rem...,1
9,Claude Code Action Official,General-purpose Claude agent for GitHub PRs an...,https://github.com/marketplace/actions/claude-...,1


In [ ]:
from google.colab import files
files.download("github_marketplace_actions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ===== NLTK FIX (Colab / Python 3.12) =====
!pip -q install nltk

import re, html
import pandas as pd
import nltk

# Download ALL required NLTK resources (includes punkt_tab)
nltk.download("punkt")
nltk.download("punkt_tab")   # <-- IMPORTANT FIX
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

STOP = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def basic_clean(text: str) -> str:
    if pd.isna(text):
        return ""
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)      # remove HTML tags
    text = re.sub(r"\s+", " ", text).strip()  # normalize spaces
    return text.lower()

def preprocess_text(text: str):
    text = basic_clean(text)

    # keep only letters/numbers/spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)

    # remove stopwords + short tokens
    tokens = [t for t in tokens if t not in STOP and len(t) > 2]

    # lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

print("Setup complete")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Setup complete


[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
import nltk
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

STOP = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text: str):
    text = basic_clean(text)

    # keep letters/numbers/spaces only
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)

    # remove stopwords + very short tokens
    tokens = [t for t in tokens if t not in STOP and len(t) > 2]

    # lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

In [ ]:
# ===== NLTK FIX (Colab / Python 3.12) =====
!pip -q install nltk

import re, html
import pandas as pd
import nltk

# Download ALL required NLTK resources (includes punkt_tab)
nltk.download("punkt")
nltk.download("punkt_tab")   # <-- IMPORTANT FIX
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

STOP = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def basic_clean(text: str) -> str:
    if pd.isna(text):
        return ""
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)      # remove HTML tags
    text = re.sub(r"\s+", " ", text).strip()  # normalize spaces
    return text.lower()

def preprocess_text(text: str):
    text = basic_clean(text)

    # keep only letters/numbers/spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)

    # remove stopwords + short tokens
    tokens = [t for t in tokens if t not in STOP and len(t) > 2]

    # lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

print("Setup complete ")

Setup complete 


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
df = pd.read_csv("github_marketplace_actions.csv")

df["product_name_clean"] = df["product_name"].map(basic_clean)
df["description_clean"]  = df["description"].map(basic_clean)

df["description_tokens"] = df["description"].map(preprocess_text)

df[["product_name", "description", "description_tokens"]].head(5)

,product_name,description,description_tokens
0,TruffleHog OSS,Find and verify leaked credentials in your sou...,"[find, verify, leaked, credential, source, code]"
1,Metrics embed,An infographics generator with 40+ plugins and...,"[infographics, generator, plugins, 300, option..."
2,yq - portable yaml processor,"create, read, update, delete, merge, validate ...","[create, read, update, delete, merge, validate..."
3,Super-Linter,Super-linter is a ready-to-run collection of l...,"[super, linter, ready, run, collection, linter..."
4,Rebuild Armbian and Kernel,"Support Amlogic, Rockchip and Allwinner boxes","[support, amlogic, rockchip, allwinner, box]"


In [ ]:
report = {}

# 1) Completeness: missing values
report["missing_product_name"] = int(df["product_name"].isna().sum() + (df["product_name"].astype(str).str.strip() == "").sum())
report["missing_description"]  = int(df["description"].isna().sum()  + (df["description"].astype(str).str.strip() == "").sum())
report["missing_url"]          = int(df["url"].isna().sum()          + (df["url"].astype(str).str.strip() == "").sum())
report["missing_page"]         = int(df["page"].isna().sum())

# 2) URL formatting validity (basic)
report["bad_url_format"] = int((~df["url"].astype(str).str.startswith("https://github.com/marketplace/actions/")).sum())

# 3) Duplicate detection
report["duplicate_urls"] = int(df.duplicated(subset=["url"]).sum())
report["duplicate_name_desc"] = int(df.duplicated(subset=["product_name_clean", "description_clean"]).sum())

# 4) Basic consistency checks
report["non_positive_page"] = int((df["page"] <= 0).sum())

report_df = pd.DataFrame([report]).T.reset_index()
report_df.columns = ["check", "count"]
report_df

,check,count
0,missing_product_name,0
1,missing_description,1
2,missing_url,0
3,missing_page,0
4,bad_url_format,0
5,duplicate_urls,0
6,duplicate_name_desc,0
7,non_positive_page,0


In [ ]:
# Remove duplicate URLs (keep first)
df = df.drop_duplicates(subset=["url"]).copy()

# Drop rows missing key fields
df["product_name"] = df["product_name"].fillna("").astype(str).str.strip()
df["url"] = df["url"].fillna("").astype(str).str.strip()
df = df[(df["product_name"] != "") & (df["url"] != "")].copy()

# Recompute cleaned columns after filtering
df["product_name_clean"] = df["product_name"].map(basic_clean)
df["description_clean"]  = df["description"].fillna("").map(basic_clean)
df["description_tokens"] = df["description_clean"].map(preprocess_text)

df.shape

(1015, 7)

In [ ]:
df.to_csv("github_marketplace_actions_cleaned.csv", index=False)

final_report = {
    "rows_final": len(df),
    "unique_urls_final": df["url"].nunique(),
    "missing_desc_final": int((df["description_clean"].astype(str).str.strip() == "").sum()),
}
pd.DataFrame([final_report])

,rows_final,unique_urls_final,missing_desc_final
0,1015,1015,1


#Question 5 (20 points)

PART 1:
Web Scrape  tweets from Twitter using the Tweepy API, specifically targeting hashtags related to subtopics (machine learning or artificial intelligence.)
The extracted data includes the tweet ID, username, and text.

Part 2:
Perform data cleaning procedures

A final data quality check ensures the completeness and consistency of the dataset. The cleaned data is then saved into a CSV file for further analysis.


**Note**

1.   Follow tutorials provided in canvas to obtain api keys. Use ChatGPT to get the code. Make sure the file is downloaded and saved.
2.   Make sure you divide GPT code as shown in tutorials, dont make multiple requestes.


In [ ]:
!pip install tweepy pandas

In [ ]:
import tweepy
import pandas as pd
import re

In [ ]:
# CELL 3: OAuth1 Authentication

import tweepy

API_KEY = "EXM6vCqWcIVC3eo9uFkbQMiSC"
API_SECRET = "QLqxBgubJSTfvHFcN1WzSPMUB9uxkZTm3mh3TQdCNcV6E2QRMEx"
ACCESS_TOKEN = "2028311047086772224-P3nkSWkX0ppWPSNZO2gDJkViqMebTk"
ACCESS_TOKEN_SECRET = "xnmnvKHzoxRR9W4D8HcLPzJH3eIgZan2YyPh7kde0fTHi"

auth = tweepy.OAuth1UserHandler(
    API_KEY,
    API_SECRET,
    ACCESS_TOKEN,
    ACCESS_TOKEN_SECRET
)

api = tweepy.API(auth, wait_on_rate_limit=True)

print("Connected via OAuth1")

Connected via OAuth1


In [ ]:
# CELL 4: Search tweets using API v1.1 (FREE)

query = "#MachineLearning OR #ArtificialIntelligence OR #AI OR #DeepLearning"
max_tweets = 100

tweets = tweepy.Cursor(
    api.search_tweets,
    q=query,
    lang="en",
    tweet_mode="extended"
).items(max_tweets)

In [ ]:
data = []

for tweet in tweets:
    data.append([
        tweet.id,
        tweet.user.screen_name,
        tweet.full_text
    ])

import pandas as pd
df = pd.DataFrame(data, columns=["tweet_id", "username", "text"])
df.head()

# Mandatory Question (5 points)

Provide your thoughts on the assignment. What did you find challenging, and what aspects did you enjoy? Your opinion on the provided time to complete the assignment.

The task was rather difficult to me due to the time-consuming nature of the task and the length of the questions. Particularly, the third question required nearly an hour to even read and digest the paper before I was able to work on it. It was necessary to be careful about the requirements and to arrange this information. Nevertheless, I liked different type of work as opposed to typical coding or workshop based activities. It tested my ability to solve problems and provided me with an opportunity to learn to work with real-life data and tools. In general, the assignment was a good learning experience, although it took more time than I had anticipated.